# VisionTrack — Analysis Notebook

## Experiment 01: Pre-trained YOLOv8n vs fine-tuned `best.pt`

**Date:** 2026-10-19. **Goal:** investigate how model choice and detection confidence affect line-crossing counts and reported processing speed in two Streamlit video streams, **Crosswalk** and **Mall**.

**Evidence:** four screenshots recorded after processing the videos. Source observations are preserved in `../reports/experiments/model_comparison.csv` and `../reports/experiments/screenshots/`.

> ⚠️ These are observed *LineZone counters*, not ground-truth person counts. Without manual verification of crossings, higher IN/OUT numbers **do not prove** greater accuracy. FPS and latency are values shown in the UI, not a separately controlled benchmark.

### Experimental setup

- Models: standard COCO-pretrained `yolov8n.pt` versus user fine-tuned `models/checkpoints/best.pt` (single class: `person`).
- Confidence thresholds: **0.50** and **0.35**.
- Input video sources: `data/raw_videos/crosswalk.mp4`, `data/raw_videos/mall.mp4`.
- Tracking: `supervision.ByteTrack`; counting: `supervision.LineZone`.
- Tracking and Counting toggles were enabled in the screenshots.
- Metrics recorded from final Streamlit display: IN, OUT, FPS, latency in milliseconds.
- Hardware, processing time, warm-up conditions, and repeated trials were **not independently measured**, so FPS comparisons are indicative only.

**Reproducibility notes:** run each pair from the beginning with a newly initialized tracker/LineZone. Keep videos, ROI line placement, resizing and application configuration identical. Repeat runs and measure FPS independently for rigorous benchmarking.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
CSV_PATH = PROJECT_ROOT / 'reports' / 'experiments' / 'model_comparison.csv'
df = pd.read_csv(CSV_PATH)
df['model_label'] = df['model'].str.replace('.pt', '', regex=False) + ' / ' + df['confidence'].map(lambda x: f'{x:.2f}')
display(df.drop(columns=['screenshot']))

### Recorded results

| Model | Confidence | Crosswalk IN | Crosswalk OUT | Mall IN | Mall OUT | Crosswalk FPS | Mall FPS | Crosswalk latency (ms) | Mall latency (ms) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| best.pt | 0.50 | 0 | 4 | 7 | 7 | 49.6 | 50.0 | 20.2 | 20.0 |
| best.pt | 0.35 | 0 | 9 | 7 | 10 | 48.4 | 47.6 | 20.7 | 21.0 |
| yolov8n.pt | 0.50 | 3 | 16 | 9 | 12 | 45.0 | 44.1 | 22.2 | 22.7 |
| yolov8n.pt | 0.35 | 5 | 21 | 8 | 12 | 47.8 | 46.9 | 20.9 | 21.3 |

In [ ]:
ax = df.plot(x='model_label', y=['crosswalk_total_crossings', 'mall_total_crossings'], kind='bar', figsize=(10, 5))
ax.set_title('Recorded line crossings (IN + OUT); NOT accuracy')
ax.set_xlabel('Model / confidence')
ax.set_ylabel('Counted crossings')
ax.legend(['Crosswalk', 'Mall'])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df.plot(x='model_label', y=['crosswalk_fps', 'mall_fps'], kind='bar', ax=ax)
ax.set_title('UI-reported FPS: single observed value per run')
ax.set_xlabel('Model / confidence')
ax.set_ylabel('Displayed FPS')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Interpretation

1. Standard `yolov8n.pt` produced more recorded crossings than `best.pt` at both matching confidence thresholds. At **0.35**, Crosswalk recorded **26** vs **9**, while Mall recorded **20** vs **17**.
2. Lowering confidence from **0.50** to **0.35** increased recorded crossings on Crosswalk for both models; on Mall the totals changed from **14 to 17** for fine-tuned weights and **21 to 20** for the standard model.
3. The screenshots suggest the pretrained model identifies more distant people, but a visual impression is **not a formal detection metric**.
4. The pretrained model is a reasonable **provisional** application choice. Final selection requires counting against manually annotated crossings plus precision/recall/F1 on a labeled dataset.
5. UI-reported FPS ranged from **44.1 to 50.0** across these runs, but a single screenshot value cannot establish reliable throughput or speedup.

### Screenshot evidence

These files are copies of screenshots supplied during development; they document the final UI state of each experimental run.

**Fine-tuned best.pt, 0.50**

![](../reports/experiments/screenshots/Screenshot%202026-10-19%20at%2016.38.26.png)

**Fine-tuned best.pt, 0.35**

![](../reports/experiments/screenshots/Screenshot%202026-10-19%20at%2016.42.25.png)

**Pre-trained yolov8n.pt, 0.50**

![](../reports/experiments/screenshots/Screenshot%202026-10-19%20at%2016.46.00.png)

**Pre-trained yolov8n.pt, 0.35**

![](../reports/experiments/screenshots/Screenshot%202026-10-19%20at%2016.48.07.png)

### Next validation steps (not yet done)

- [ ] Manually label each true IN and OUT crossing for a fixed video segment and line direction.
- [ ] Compare predicted crossing events to ground truth (missed, extra, correct events).
- [ ] Evaluate detection precision, recall and F1 on labeled held-out frames, keeping IoU/confidence policy fixed.
- [ ] Repeat benchmark multiple times and measure per-stream effective FPS and end-to-end latency, report hardware and mean ± variation.
- [ ] Record fine-tuning training settings, learning curves, and validation scores in the separate training analysis.
- [ ] Compare original weights and ONNX quantized model on matching inputs; document accuracy and speed differences.

**Important:** do not enter these crossing totals as `detection_precision`, `detection_recall`, or `f1_score` in `reports/performance_metrics.json`.